# Deploy NPS Agent to RHOAI

This notebook deploys the NPS Agent ([`npsagent.py`](./npsagent.py)) to OpenShift AI with MLflow tracing.

## From Local Development to OpenShift Deployment

In the [Evaluate notebook](../1_develop/2_evaluate.ipynb) we built and tested the NPS Agent locally — running it inside a Jupyter cell, calling NPS tools over MCP, and evaluating the results with MLflow scorers.

Now we'll take that **exact same agent logic** and deploy it as a live HTTP endpoint on **Red Hat OpenShift AI (RHOAI)**. The core agent code stays almost identical; we just add a thin serving layer so MLflow can host it as a REST API.

Here's what changes (and what doesn't):

| | Local (evaluate notebook) | Deployed (this notebook) |
|---|---|---|
| **Agent logic** | `run_nps_agent()` with `MCPServerStdio` | Same `run_nps_agent()` with `MCPServerStdio` |
| **MCP server** | Spawned via `uv run fastmcp run` | Same — `uv` is installed in the container |
| **Serving** | Direct `await` in a notebook cell | Wrapped in MLflow `ResponsesAgent` and served over HTTP |
| **Tracing** | `mlflow.openai.autolog()` to local MLflow | Same autolog, but traces go to RHOAI MLflow |
| **Auth** | N/A | RHOAI workspace header for MLflow routing |
| **Infrastructure** | Your laptop | OpenShift pod (s2i build from Git) |


### Cluster Setup

Before continuing, you'll need an OpenShift cluster with RHOAI and MLflow already configured. Follow the [Cluster Setup Guide](https://docs.google.com/document/d/1ZzuGAY1gSamOLsznwbaL7xFkJ1JjHWpnyjk0tV12YVg/edit?tab=t.0#heading=h.jgt5ddlrwyvc) to get your environment ready.

---

## Deployment Files Overview

The `2_deploy/` directory is self-contained — everything OpenShift needs to build and run the agent:

| File | What it does |
|---|---|
| [`npsagent.py`](./npsagent.py) | The agent logic (identical to the evaluate notebook) wrapped in an MLflow `ResponsesAgent` so it can serve HTTP requests via `POST /invocations` |
| [`nps_mcp_server.py`](./nps_mcp_server.py) | FastMCP server exposing NPS API tools. Spawned on-demand by `run_nps_agent` via `uv run fastmcp run` — not a long-running process |
| [`app.sh`](./app.sh) | Container entry point. Packages `npsagent.py` with `mlflow.pyfunc.save_model`, then starts `mlflow models serve` on port 8080 (5 min timeout) |
| [`requirements.txt`](./requirements.txt) | Python dependencies installed during the s2i build |
| [`nps-agent.yaml`](./nps-agent.yaml) | OpenShift Template that creates a **BuildConfig** (s2i from `deploydemo` branch using `python:3.12-ubi9`), **ImageStream**, **Deployment** (with secret injection + readiness/liveness probes on `/ping`), **Service** (port 8080), and **Route** (HTTPS with 5 min HAProxy timeout) — all in one `oc process` call |
| [`.s2i/environment`](./.s2i/environment) | Single line: `APP_SCRIPT=app.sh` — tells the [s2i](https://github.com/openshift/source-to-image) builder to use `app.sh` instead of the default Python entrypoint |

---

## What's Different in the Deployed Version?

The core `run_nps_agent` in [`npsagent.py`](./npsagent.py) is identical to the evaluate notebook. The only additions are:

1. **`NPSResponsesAgent`** — wraps the agent for MLflow HTTP serving
2. **RHOAI workspace header** — routes traces to the correct project
3. **`mlflow.openai.autolog()`** — auto-traces every LLM call and tool invocation

### `NPSResponsesAgent` — MLflow Serving Wrapper

MLflow serves models over HTTP via a `POST /invocations` endpoint. To plug our agent into this, we subclass [`ResponsesAgent`](https://mlflow.org/docs/latest/python_api/mlflow.pyfunc.html) — MLflow's standard interface for chat-style models. The `predict` method receives the request, passes `request.input` directly to `run_nps_agent`, and returns the result in MLflow's Responses format. No message extraction needed — `Runner.run()` accepts both strings and Responses API input items. This is what turns our agent into a deployable HTTP service.

Authentication to RHOAI's MLflow is handled by `MLFLOW_TRACKING_AUTH=kubernetes` in [`nps-agent.yaml`](./nps-agent.yaml) — no auth code needed in the agent itself. This requires `mlflow>=3.10.0`, so we use the [Red Hat build of MLflow](https://github.com/opendatahub-io/mlflow) from GitHub.

See [`npsagent.py`](./npsagent.py) for the full code.

---

## Prerequisites

Before you begin, make sure you have:

- An OpenShift cluster with RHOAI and MLflow configured
- The `oc` CLI installed and logged in to your cluster
- An [OpenAI API key](https://platform.openai.com/api-keys)
- An [NPS API key](https://www.nps.gov/subjects/developer/get-started.htm) (free and instant)
- A `.env` file in the repository root with your `OPENAI_API_KEY` and `NPS_API_KEY`

In [ ]:
import os
import subprocess
from dotenv import load_dotenv, find_dotenv

# Load environment variables from .env file (searches current and parent directories)
load_dotenv(find_dotenv())

## OpenAI Environment Variables
os.environ.setdefault("OPENAI_API_KEY", "")
os.environ.setdefault("OPENAI_BASE_URL", "https://api.openai.com/v1")
os.environ.setdefault("OPENAI_MODEL_NAME", "gpt-4o")

## NPS API Key
os.environ.setdefault("NPS_API_KEY", "")

# Check that required vars are set
required_vars = ["OPENAI_API_KEY", "OPENAI_BASE_URL", "OPENAI_MODEL_NAME", "NPS_API_KEY"]
if any(not os.getenv(var) for var in required_vars):
    raise ValueError("One or more required environment variables are not set. Check your .env file.")

## Step 1 — Create an OpenShift Project

Each deployment lives in its own OpenShift namespace. Set yours below — use `nps-agent-<yourname>` to avoid conflicts with other users on the same cluster. `MLFLOW_TRACKING_URI` is read from your `.env` file.

<center>
<img src="./images/projects.png" alt="OpenShift Projects" width="75%"/>
<br>
<em>OpenShift Console — Projects view</em>
</center>

In [ ]:
NAMESPACE = "<yourname>"

MLFLOW_TRACKING_URI = os.getenv("MLFLOW_TRACKING_URI", "")
MLFLOW_EXPERIMENT_NAME = "nps-agent"

In [ ]:
!oc new-project {NAMESPACE}

## Step 2 — Create the Secret with API Keys

Store your API keys as an OpenShift Secret so the running pod can access them without baking credentials into the image. `OPENAI_BASE_URL` and `OPENAI_MODEL_NAME` are optional — they default to the OpenAI API and `gpt-4o-mini` if not set in your `.env`.

<center>
<img src="./images/secrets.png" alt="OpenShift Secrets" width="75%"/>
<br>
<em>OpenShift Console — Secret details for nps-agent-secrets</em>
</center>

In [ ]:
!oc create secret generic nps-agent-secrets \
  --from-literal=OPENAI_API_KEY="{os.getenv('OPENAI_API_KEY')}" \
  --from-literal=NPS_API_KEY="{os.getenv('NPS_API_KEY')}" \
  --from-literal=OPENAI_BASE_URL="{os.getenv('OPENAI_BASE_URL', 'https://api.openai.com/v1')}" \
  --from-literal=OPENAI_MODEL_NAME="{os.getenv('OPENAI_MODEL_NAME', 'gpt-4o-mini')}" \
  -n {NAMESPACE}

## Step 3 — Grant Service Account Access

The pod's default service account needs `admin` access to the project namespace so it can authenticate with the MLflow tracking server via `MLFLOW_TRACKING_AUTH=kubernetes`.

> **Note:** The `MLFLOW_TRACKING_URI` must point to an **in-cluster MLflow route** (not the data-science-gateway). If your cluster doesn't already have one, create a route in the `redhat-ods-applications` namespace targeting the `mlflow` service on port `8443` with TLS re-encrypt. See the [Cluster Setup Guide](https://docs.google.com/document/d/1ZzuGAY1gSamOLsznwbaL7xFkJ1JjHWpnyjk0tV12YVg/edit?tab=t.0#heading=h.jgt5ddlrwyvc) for details.

In [ ]:
!oc adm policy add-role-to-user admin system:serviceaccount:{NAMESPACE}:default -n {NAMESPACE}

## Step 4 — Apply the OpenShift Template

Process [`nps-agent.yaml`](./nps-agent.yaml) with your cluster values and apply all resources (see [Deployment Files](#Deployment-Files-Overview) above for what each resource does).

<center>
<img src="./images/service.png" alt="OpenShift Services" width="75%"/>
<br>
<em>OpenShift Console — Service created for the nps-agent on port 8080</em>
</center>

In [ ]:
!oc process -f ./nps-agent.yaml \
  -p NAMESPACE="{NAMESPACE}" \
  -p MLFLOW_TRACKING_URI="{MLFLOW_TRACKING_URI}" \
  -p MLFLOW_EXPERIMENT_NAME="{MLFLOW_EXPERIMENT_NAME}" \
  | oc apply -f -

## Step 5 — Wait for the s2i Build

The BuildConfig triggers automatically. Watch until you see **"Push successful"**.

In [ ]:
!oc logs -f build/nps-agent-1 -n {NAMESPACE}

## Step 6 — Verify the Pod is Running

Once the build completes and the image is pushed, the Deployment rolls out a pod. Check that it's in `Running` state before continuing.

<center>
<img src="./images/verifypod.png" alt="OpenShift Pods" width="75%"/>
<br>
<em>OpenShift Console — Pod list showing the completed build and running agent</em>
</center>

In [ ]:
!oc get pods -n {NAMESPACE}

## Step 7 — Get the Route URL

The OpenShift Route exposes the agent pod as a public HTTPS endpoint. Grab the hostname so we can call it.

<center>
<img src="./images/route.png" alt="OpenShift Routes" width="75%"/>
<br>
<em>OpenShift Console — Route with the public HTTPS endpoint</em>
</center>

In [ ]:
ROUTE_HOST = subprocess.check_output(
    ["oc", "get", "route", "nps-agent", "-n", NAMESPACE, "-o", "jsonpath={.spec.host}"]
).decode().strip()

AGENT_URL = f"https://{ROUTE_HOST}"
print(f"Agent URL: {AGENT_URL}")

## Step 8 — Test the Agent

Send a sample question to the deployed agent's `/invocations` endpoint and display the response.

In [ ]:
import requests
print(AGENT_URL)
resp = requests.post(f"{AGENT_URL}/invocations", json={
    "input": [{"role": "user", "content": "What national parks are in California?"}]
}, timeout=300)

print(resp.json()["output"][0]["content"][0]["text"] if resp.ok else f"Error: {resp.text}")

## Step 9 — View Traces in MLflow

View your traces in your MLflow cluster at `<MLFLOW_TRACKING_URI>/#/experiments`.

<center>
<img src="./images/mlflowtraces.png" alt="MLflow Traces" width="75%"/>
<br>
<em>MLflow trace view showing agent execution, tool calls, and response</em>
</center>

## Rebuilding After Code Changes

Push changes to the `main` branch, then trigger a new build:

In [ ]:
# Uncomment to trigger a rebuild after pushing code changes
# !oc start-build nps-agent -n {NAMESPACE}
# !oc logs -f build/nps-agent-2 -n {NAMESPACE}

## Cleanup

To delete everything and start from scratch:

In [ ]:
# Uncomment to delete everything and start from scratch
#!oc delete project {NAMESPACE}